In [1]:
import sys
import os
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 确保 src/ 包可被导入
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, root_dir)

from config import COAL_TYPES, TRAIN_DIR, TEST_DIR, AUX_COLS, ALPHAS
from src.data   import load_labels, load_coal_spectra
from src.submit import pack_submission

In [2]:
label_map, aux_map = load_labels()
coal_type = COAL_TYPES[0]
train_data = load_coal_spectra(TRAIN_DIR, coal_type, label_map, aux_map)

In [3]:
train_data.keys()

dict_keys(['spectra', 'stats', 'labs', 'lrel', 'rats', 'names', 'targets', 'aux', 'groups', 'n_batches'])

In [4]:
groups = np.array(train_data['groups'])
names = train_data['names']

In [5]:
for group_idx in np.unique(groups):
    mask = np.where(groups == group_idx)
    name = names[mask[0][0]]
    print(f"Batch name: {name}, {len(mask[0])} spectreal data found")

Batch name: 赵固一矿豫焦末煤11月19日, 12 spectreal data found
Batch name: 赵固一矿豫焦末煤11月27日, 11 spectreal data found
Batch name: 赵固一矿豫焦末煤11月30日, 13 spectreal data found
Batch name: 赵固一矿豫焦末煤11月7日, 13 spectreal data found
Batch name: 赵固一矿豫焦末煤11月9日, 13 spectreal data found
Batch name: 赵固一矿豫焦末煤1月13日, 14 spectreal data found
Batch name: 赵固一矿豫焦末煤1月14日, 12 spectreal data found
Batch name: 赵固一矿豫焦末煤1月27日, 13 spectreal data found
Batch name: 赵固一矿豫焦末煤1月28日, 12 spectreal data found
Batch name: 赵固一矿豫焦末煤2月12日, 15 spectreal data found
Batch name: 赵固一矿豫焦末煤2月2日, 12 spectreal data found


In [6]:
from src.features import compute_features
inorm_mat = compute_features(train_data)
labs = train_data['labs']
lrel = train_data['lrel']
rats = train_data['rats']

In [7]:
labs.shape, lrel.shape, rats.shape

((140, 11), (140, 11), (140, 4))

In [8]:
inten_meanl, hand_meanl = [], []
for group_idx in np.unique(groups):
    mask = np.where(groups == 1)
    inorm_mean = inorm_mat[mask].mean(axis=0)
    labs_mean = labs[mask].mean(axis=0)
    lrel_mean = lrel[mask].mean(axis=0)
    rats_mean = rats[mask].mean(axis=0)
    hand = np.hstack((labs_mean, lrel_mean, rats_mean))
    inten_meanl.append(inorm_mean)
    hand_meanl.append(hand)

np.array(inten_meanl).shape, np.array(hand_meanl).shape

((11, 7305), (11, 26))

In [9]:
def compute_batch_features(data, batch_ids):
    """
    Aggregates shot-level features into robust batch-level representations.
    
    batch_ids: Array of shape (n_shots,) containing the batch/sample ID for each shot.
    """
    raw_spectra = data['spectra']
    unique_batches = np.unique(batch_ids)
    
    batch_inorms = []
    batch_hand_feats = []
    
    for b_id in unique_batches:
        mask = (batch_ids == b_id)
        
        # A. Batch-Averaged Spectrum (Averages out shot-to-shot continuum noise)
        mean_inorm = inorm_mat[mask].mean(axis=0)
        batch_inorms.append(mean_inorm)
        
        # B. Batch Statistics for Handcrafted Features
        # Compute Mean, Median, and Std across shots in this batch
        mean_rats   = data['rats'][mask].mean(axis=0)
        median_rats = np.median(data['rats'][mask], axis=0)
        std_rats    = data['rats'][mask].std(axis=0)
        
        mean_lrel   = data['lrel'][mask].mean(axis=0)
        median_lrel = np.median(data['lrel'][mask], axis=0)
        
        # Combine into a single rich batch-level handcrafted vector
        b_hand = np.hstack([
            mean_lrel, median_lrel, 
            mean_rats, median_rats, std_rats
        ])
        batch_hand_feats.append(b_hand)

    batch_inorm_mat = np.array(batch_inorms, dtype=np.float32)
    batch_hand_mat  = np.array(batch_hand_feats, dtype=np.float32)

    return batch_inorm_mat, batch_hand_mat, unique_batches

In [10]:
batch_inorm_mat, batch_hand_mat, _ = compute_batch_features(train_data, groups)

In [11]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from config import RANDOM_STATE

N_PCA_MAX = 30
n_batches = train_data["n_batches"]

# PCA
scaler_spec = StandardScaler()
spec_scaled = scaler_spec.fit_transform(batch_inorm_mat)
n_pca = min(N_PCA_MAX, n_batches - 1)
pca = PCA(n_components=n_pca, random_state=RANDOM_STATE)
spec_pca = pca.fit_transform(spec_scaled)

In [19]:
X = np.hstack((spec_pca, batch_hand_mat))
scaler_hand = StandardScaler()
X = scaler_hand.fit_transform(X)
X.shape

(11, 44)

In [13]:
# Get the first occurrence of each batch
_, unique_indices = np.unique(groups, return_index=True)
target_batch = train_data['targets'][unique_indices]
aux_batch = train_data['aux'][unique_indices]
target_batch.shape, aux_batch.shape

((11,), (11, 4))

In [14]:
from sklearn.model_selection import KFold
cv_splits = []
kf = KFold(n_splits=5)
for train_idx, val_idx in kf.split(target_batch):
    cv_splits.append((train_idx, val_idx))

len(cv_splits)

5

In [42]:
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor

pred_aux_oof = np.zeros_like(aux_batch)
for aux_id in range(aux_batch.shape[1]):
    y_aux = aux_batch[:,aux_id]

    ridge = RidgeCV(alphas=ALPHAS)
    oof = np.zeros_like(y_aux)
    for tr_idx, val_idx in cv_splits:
        ridge.fit(X[tr_idx],y_aux[tr_idx])
        oof[val_idx] = ridge.predict(X[val_idx])

    pred_aux_oof[:,aux_id] = oof
    rmse = np.sqrt(np.mean((oof - y_aux) ** 2))
    per = rmse / np.mean(y_aux) * 100
    print(f"Batch level prediction for {AUX_COLS[aux_id]} | RMSE: {rmse: .2f} | Percentage error: {per: .2f}%")

Batch level prediction for 全水分 | RMSE:  1.30 | Percentage error:  13.90%
Batch level prediction for 灰分 | RMSE:  3.69 | Percentage error:  16.83%
Batch level prediction for 氢 | RMSE:  0.08 | Percentage error:  3.72%
Batch level prediction for 硫 | RMSE:  0.07 | Percentage error:  16.81%


In [40]:
# features for stage 2
scaler_s2 = StandardScaler()
X_s2 = np.hstack([X, scaler_s2.fit_transform(pred_aux_oof)])
X_s2.shape

(11, 48)

In [43]:
y_pred = np.zeros_like(target_batch)
for tr_idx, val_idx in cv_splits:
    m = RidgeCV(alphas=ALPHAS)
    m.fit(X_s2[tr_idx], target_batch[tr_idx])
    val_pred = m.predict(X_s2[val_idx])
    y_pred[val_idx] = val_pred

y_rmse = np.sqrt(np.mean((y_pred - target_batch) ** 2))
print(f"Calorific value RMSE: {y_rmse: .2f}")

Calorific value RMSE:  438.74


This does not improve the prediction. Let's try a hybrid strategy: shot level prediction in stage one and batch level prediction in stage 2.

## Hybrid architectrue

In [48]:
from src.features import build_feature_matrix
from src.model import get_cv_splits

X_spec, scaler_spec, pca, scaler_hand = build_feature_matrix(
        train_data, n_batches, fit=True)

y          = train_data['targets']
aux        = train_data['aux']

splits = get_cv_splits(groups, n_batches)

In [49]:
predicted_aux_oof = np.zeros_like(aux, dtype=np.float32)
for col_idx, col_name in enumerate(AUX_COLS):
    y_aux = aux[:,col_idx]
    
    m = RidgeCV(alphas=ALPHAS)
    oof = np.zeros(len(y_aux))
    # Foldwise model for intermediate prediction
    for tr_idx, val_idx in splits:
        m.fit(X_spec[tr_idx], y_aux[tr_idx])
        oof[val_idx] = m.predict(X_spec[val_idx])
    predicted_aux_oof[:, col_idx] = oof

    # Diagnostic
    rmse = np.sqrt(np.mean((oof - y_aux) ** 2))
    print(f"Aux variable {col_name}")
    print(f"  RMSE: {rmse: .4f}")
    print(f"  Percentage error: {rmse / np.mean(y_aux) * 100: .2f}%")

Aux variable 全水分
  RMSE:  1.4306
  Percentage error:  15.38%
Aux variable 灰分
  RMSE:  3.9818
  Percentage error:  18.28%
Aux variable 氢
  RMSE:  0.0872
  Percentage error:  3.83%
Aux variable 硫
  RMSE:  0.0603
  Percentage error:  13.73%


In [53]:
predicted_aux_oof.shape

(140, 4)

In [58]:
# construct the batch mean of the predicted value
mean_auxl = []
for group_idx in np.unique(groups):
    mask = (groups == group_idx)
    mean_aux = predicted_aux_oof[mask].mean(axis = 0)
    mean_auxl.append(mean_aux)

mean_auxl = np.array(mean_auxl)
# Diagnostic
rmse = np.sqrt(np.mean((mean_auxl - aux_batch) ** 2, axis = 0))
for idx in range(len(AUX_COLS)):
    print(f"Aux variable {AUX_COLS[idx]}")
    print(f"  RMSE: {rmse[idx]: .4f}")
    print(f"  Percentage error: {rmse[idx] / np.mean(aux_batch[:,idx]) * 100: .2f}%")


Aux variable 全水分
  RMSE:  1.3911
  Percentage error:  14.90%
Aux variable 灰分
  RMSE:  3.5633
  Percentage error:  16.25%
Aux variable 氢
  RMSE:  0.0774
  Percentage error:  3.41%
Aux variable 硫
  RMSE:  0.0527
  Percentage error:  11.98%


In [ ]:
# features for stage 2
scaler_s2 = StandardScaler()
mean_auxl_norm = scaler_s2.fit_transform(mean_auxl)
X_s2 = np.hstack([X, mean_auxl_norm])
mean_auxl_norm.shape, X_s2.shape

((11, 4), (11, 48))

In [71]:
y_pred = np.zeros_like(target_batch)
for tr_idx, val_idx in cv_splits:
    m = RidgeCV(alphas=ALPHAS)
    m.fit(X_s2[tr_idx], target_batch[tr_idx])
    val_pred = m.predict(X_s2[val_idx])
    y_pred[val_idx] = val_pred

y_rmse = np.sqrt(np.mean((y_pred - target_batch) ** 2))
print(f"Calorific value RMSE: {y_rmse: .2f}")

Calorific value RMSE:  437.77


In [67]:
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

# 2. Define a dictionary of powerful, diverse models
# Note: SVR and PLS benefit heavily from pre-scaled features, so we wrap SVR in a pipeline.
models = {
    "Ridge (Baseline)": RidgeCV(alphas=ALPHAS),

    "Lasso": LassoCV(alphas=ALPHAS),
    
    # GridSearch dynamically finds the optimal latent variables for your specific spectra
    "PLS (Tuned)": GridSearchCV(
        PLSRegression(), 
        param_grid={"n_components": [5, 10, 15, 20, 25, 30]}, 
        cv=3
    ),
    
    # ElasticNet is far more stable on collinear LIBS spectra than pure Lasso
    "ElasticNet": ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9], cv=3, max_iter=5000),
    
    # SVR with RBF kernel is fantastic for non-linear matrix effects
    "SVR (RBF Kernel)": make_pipeline(
        StandardScaler(), 
        SVR(C=10.0, epsilon=0.01)
    ),
    
    # Tree models excel at finding indirect proxies (like Fe lines representing FeS2)
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42),
    
    "Gradient Boosting": HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, random_state=42)
}

# 3. Automated Benchmarking Loop
results = {}

print(f"=== Benchmarking Non-linear Models ===")
print(f"Mean Target Value: {np.mean(target_batch):.4f}\n")

for name, model in models.items():
    oof_preds = np.zeros(len(target_batch))
    
    # Foldwise fitting and out-of-fold prediction
    for tr_idx, val_idx in cv_splits:
        X_tr, y_tr = X_s2[tr_idx], target_batch[tr_idx]
        X_val = X_s2[val_idx]
        
        model.fit(X_tr, y_tr)
        oof_preds[val_idx] = model.predict(X_val)
        
    # Calculate mathematically correct RMSE and Relative Error
    rmse = np.sqrt(np.mean((oof_preds - target_batch) ** 2))
    rel_error = (rmse / np.mean(target_batch)) * 100
    
    results[name] = {"RMSE": rmse, "RelError": rel_error, "OOF": oof_preds}
    
    # Print real-time diagnostic
    print(f"Model: {name:<20} | RMSE: {rmse:.4f} | Relative Error: {rel_error:.2f}%")
    
    # If it's PLS, let's see how many components it actually chose on the last fold
    if name == "PLS (Tuned)":
        print(f"  -> Optimal PLS components chosen: {model.best_params_['n_components']}")

=== Benchmarking Non-linear Models ===
Mean Target Value: 5700.3636

Model: Ridge (Baseline)     | RMSE: 437.7712 | Relative Error: 7.68%
Model: Lasso                | RMSE: 432.3410 | Relative Error: 7.58%
Model: PLS (Tuned)          | RMSE: 543.3287 | Relative Error: 9.53%
  -> Optimal PLS components chosen: 5
Model: ElasticNet           | RMSE: 446.0018 | Relative Error: 7.82%
Model: SVR (RBF Kernel)     | RMSE: 403.0379 | Relative Error: 7.07%
Model: Random Forest        | RMSE: 452.8031 | Relative Error: 7.94%
Model: Gradient Boosting    | RMSE: 431.5158 | Relative Error: 7.57%
